# Servidor Cloud

**Autor:** Jose Ayala  
**Asignatura:** Análisis Multivariado y Modelos Estocásticos  
**Tema:** Simulación de Colas M/M/1 y Exploración Visual Multivariada


## 1. Importación de Librerías y Configuración

En esta sección se importan las librerías esenciales para realizar la simulación numérica (`numpy`), estructurar y manipular los datos obtenidos (`pandas`), y generar visualizaciones estadísticas y multivariadas avanzadas (`matplotlib` y `seaborn`).


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Configuración de estilo visual premium
sns.set_theme(style="whitegrid")
plt.rcParams.update({
    'font.size': 11,
    'axes.labelsize': 12,
    'axes.titlesize': 14,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'figure.titlesize': 16
})


## 2. Definición del Modelo de Simulación de Cola M/M/1

La función `simular_cola_mm1` implementa la simulación de eventos discretos para una cola **M/M/1** (llegadas y servicios markovianos con un único servidor) a partir de tasas de llegada ($\lambda$) y servicio ($\mu$).

Calcula métricas clave individuales y promedios:
- **$W_q$**: Tiempo promedio de espera en la cola.
- **$W$**: Tiempo promedio en el sistema (espera + servicio).
- **$Var(W)$**: Varianza del tiempo en el sistema.
- **$\rho$**: Factor de utilización del servidor.
- **$L$**: Número promedio de clientes en el sistema (utilizando la Ley de Little: $L = \lambda \cdot W$).


In [ ]:
def simular_cola_mm1(lam, mu, num_peticiones=1000):
    """
    Simula una cola M/M/1 y retorna métricas de rendimiento promedio.
    
    Parámetros:
        lam (float): Tasa de llegada (lambda).
        mu (float): Tasa de servicio (mu).
        num_peticiones (int): Número de peticiones/clientes a simular.
        
    Retorna:
        list: [lam, mu, rho, Wq, L, Var_W]
    """
    # Generación de tiempos utilizando distribución exponencial
    tiempos_interarribo = np.random.exponential(1/lam, num_peticiones)
    tiempos_servicio = np.random.exponential(1/mu, num_peticiones)
    
    # Tiempos acumulados de llegada al sistema
    tiempos_llegada = np.cumsum(tiempos_interarribo)
    
    tiempos_inicio_servicio = np.zeros(num_peticiones)
    tiempos_fin_servicio = np.zeros(num_peticiones)
    
    # Simulación del flujo de atención en el servidor cloud
    for i in range(num_peticiones):
        if i == 0:
            tiempos_inicio_servicio[i] = tiempos_llegada[i]
        else:
            tiempos_inicio_servicio[i] = max(tiempos_llegada[i], tiempos_fin_servicio[i-1])
            
        tiempos_fin_servicio[i] = tiempos_inicio_servicio[i] + tiempos_servicio[i]
    
    # Cálculo de métricas individuales de tiempos
    tiempos_espera_cola = tiempos_inicio_servicio - tiempos_llegada
    tiempos_sistema = tiempos_fin_servicio - tiempos_llegada
    
    # Promedios y varianza obtenidos en la simulación
    Wq = np.mean(tiempos_espera_cola)
    W = np.mean(tiempos_sistema)
    Var_W = np.var(tiempos_sistema)
    rho = lam / mu
    L = lam * W  # Ley de Little
    
    return [lam, mu, rho, Wq, L, Var_W]


## 3. Generación del Dataset Multivariado

Para explorar visualmente el comportamiento del servidor bajo diferentes niveles de carga, definimos **200 escenarios** con parámetros de simulación dinámicos. Para garantizar la estabilidad del sistema, fijamos un factor de utilización $\rho < 1.0$ (servidor estable sin desbordamiento de cola).


In [ ]:
# Fijar semilla para reproducibilidad de los experimentos
np.random.seed(42)
datos_experimentos = []

# Generar los 200 escenarios estables de simulación
for _ in range(200):
    # Definir capacidades y tasas aleatorias pero estables (rho < 1)
    mu_escenario = np.random.uniform(15, 30)
    rho_objetivo = np.random.uniform(0.4, 0.93)
    lam_escenario = mu_escenario * rho_objetivo
    
    # Ejecutar la simulación M/M/1 para este escenario
    metricas = simular_cola_mm1(lam_escenario, mu_escenario)
    datos_experimentos.append(metricas)


## 4. Estructuración y Guardado de Datos

En esta sección, consolidamos las métricas obtenidas en un DataFrame estructurado y lo guardamos en un archivo de formato CSV (`rendimiento_servidor_cloud.csv`) para facilitar análisis complementarios.


In [ ]:
# Creación del DataFrame de Pandas
columnas = ['Tasa_Llegada', 'Tasa_Servicio', 'Utilizacion_Rho', 'TiempoEspera_Wq', 'NumClientes_L', 'Varianza_W']
df_cloud = pd.DataFrame(datos_experimentos, columns=columnas)

# Exportación del dataset generado a archivo CSV
df_cloud.to_csv('rendimiento_servidor_cloud.csv', index=False)
print("¡Dataset 'rendimiento_servidor_cloud.csv' generado con éxito!")

# Vista preliminar de los primeros 5 escenarios
df_cloud.head()


## 5. Exploración Visual Multivariada

Realizamos un análisis visual avanzado del rendimiento de nuestro servidor cloud utilizando técnicas gráficas de visualización de datos multivariados.

### 5.1 Matriz de Diagramas de Dispersión (Pairplot)
Permite evaluar de forma rápida e integral la distribución marginal de cada variable (usando curvas estimadas por kernel KDE) y los patrones de correlación lineal o no lineal entre pares de variables.


In [ ]:
# Gráfico 1: Matriz de Diagramas de Dispersión (Pairplot)
pair_plot = sns.pairplot(
    df_cloud,
    diag_kind='kde',
    plot_kws={'alpha': 0.7, 'color': '#1f77b4', 'edgecolor': 'w'},
    diag_kws={'color': '#ff7f0e'}
)
pair_plot.fig.suptitle("Matriz de Dispersión Multivariada: Servidor Cloud", y=1.02, fontsize=14)
plt.show()


### 5.2 Análisis Cuadrivariado Combinado (Gráfico de Burbujas)
Este gráfico integra de forma simultánea cuatro variables en una única representación bidimensional, ofreciendo un panorama profundo del comportamiento estocástico del servidor:
- **Eje X (Posición Horizontal)**: Factor de Utilización del Servidor (`Utilizacion_Rho`).
- **Eje Y (Posición Vertical)**: Tiempo Promedio de Espera en Cola (`TiempoEspera_Wq`).
- **Color (Gama de Tonos)**: Varianza del Tiempo de Respuesta en el Sistema (`Varianza_W`), reflejando la predictibilidad o inestabilidad del sistema.
- **Tamaño de Burbuja**: Número Promedio de Clientes en el Sistema (`NumClientes_L`), multiplicando por escala para destacar visualmente.


In [ ]:
# Gráfico 2: Análisis Cuadrivariado Combinado (Gráfico de Burbujas)
plt.figure(figsize=(10, 6))
scatter = plt.scatter(
    df_cloud['Utilizacion_Rho'],        # Eje X
    df_cloud['TiempoEspera_Wq'],         # Eje Y
    c=df_cloud['Varianza_W'],            # Color representa la varianza del tiempo
    s=df_cloud['NumClientes_L'] * 10,    # Tamaño representa el número promedio de clientes
    cmap='viridis',
    alpha=0.8,
    edgecolors='w',
    linewidths=0.5
)

# Barra de escala de colores
cbar = plt.colorbar(scatter)
cbar.set_label('Varianza del Tiempo de Respuesta (Varianza_W)', fontsize=10)

# Etiquetas y formatos del gráfico
plt.title('Análisis de 4 Variables Simultáneas (Servidor Cloud)', fontsize=12, pad=12)
plt.xlabel('Factor de Utilización del Servidor (Utilizacion_Rho)', fontsize=10)
plt.ylabel('Tiempo Promedio de Espera en Cola (TiempoEspera_Wq)', fontsize=10)
plt.grid(True, linestyle='--', alpha=0.5)

# Mostrar gráfico final
plt.show()
